In [1]:
import calendar
from pathlib import Path

import cdsapi
import numpy as np
import pandas as pd
import xarray as xr

## Locate the project and weather folder

In [2]:
project_root = Path.cwd().resolve()

while project_root != project_root.parent:
    if (project_root / "data").exists():
        break

    project_root = project_root.parent

weather_folder = (
    project_root
    / "data"
    / "raw"
    / "weather"
    / "era5_land"
    / "2024"
)

weather_folder.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", project_root)
print("Weather folder:", weather_folder)

Project root: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence
Weather folder: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/2024


## ERA5-Land request settings

In [3]:
DATASET_NAME = "reanalysis-era5-land"

VARIABLES = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "total_precipitation",
]

TIMES = [
    f"{hour:02d}:00"
    for hour in range(24)
]

MANITOBA_AREA = [
    60.2,     # North
    -102.3,   # West
    48.8,     # South
    -89.5,    # East
]

FIRE_SEASON_MONTHS = list(
    range(4, 11)
)

client = cdsapi.Client()

print("Fire-season months:", FIRE_SEASON_MONTHS)
print("Variables:", len(VARIABLES))
print("Hours per day:", len(TIMES))

Fire-season months: [4, 5, 6, 7, 8, 9, 10]
Variables: 5
Hours per day: 24


## Reusable monthly download function

In [4]:
def get_month_days(
    year: int,
    month: int,
) -> list[str]:
    """Return all day numbers for a calendar month."""

    number_of_days = calendar.monthrange(
        year,
        month,
    )[1]

    return [
        f"{day:02d}"
        for day in range(
            1,
            number_of_days + 1,
        )
    ]


def download_era5_month(
    *,
    year: int,
    month: int,
    output_path: Path,
) -> None:
    """Download one complete month of ERA5-Land data."""

    if (
        output_path.exists()
        and output_path.stat().st_size > 1_000_000
    ):
        size_mb = (
            output_path.stat().st_size
            / 1_000_000
        )

        print(
            f"Already available: {output_path.name} "
            f"({size_mb:.2f} MB)"
        )

        return

    if output_path.exists():
        print(
            "Removing incomplete file:",
            output_path.name,
        )

        output_path.unlink()

    days = get_month_days(
        year,
        month,
    )

    request = {
        "variable": VARIABLES,
        "year": str(year),
        "month": f"{month:02d}",
        "day": days,
        "time": TIMES,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": MANITOBA_AREA,
    }

    print(
        f"Downloading {year}-{month:02d}: "
        f"{len(days)} days"
    )

    client.retrieve(
        DATASET_NAME,
        request,
        str(output_path),
    )

    size_mb = (
        output_path.stat().st_size
        / 1_000_000
    )

    print(
        f"Completed: {output_path.name} "
        f"({size_mb:.2f} MB)"
    )

## Download April through October 2024

This will automatically skip July because that file already exists.

In [5]:
monthly_hourly_paths = {}

for month in FIRE_SEASON_MONTHS:
    monthly_path = (
        weather_folder
        / (
            "era5_land_manitoba_"
            f"2024_{month:02d}_hourly.nc"
        )
    )

    monthly_hourly_paths[month] = (
        monthly_path
    )

    download_era5_month(
        year=2024,
        month=month,
        output_path=monthly_path,
    )

2026-08-03 23:16:13,899 INFO Request ID is f655bb49-c612-40fc-a68a-73062e327a64
2026-08-03 23:16:14,074 INFO status has been updated to accepted
2026-08-03 23:16:38,831 INFO status has been updated to running
2026-08-03 23:22:38,377 INFO status has been updated to successful


Completed: era5_land_manitoba_2024_04_hourly.nc (86.22 MB)


2026-08-03 23:22:47,882 INFO Request ID is a61c98e7-7176-4b00-b6c5-4acce79a035f
2026-08-03 23:22:48,457 INFO status has been updated to accepted
2026-08-03 23:23:23,262 INFO status has been updated to running
2026-08-03 23:29:13,225 INFO status has been updated to successful


Completed: era5_land_manitoba_2024_05_hourly.nc (93.89 MB)


2026-08-03 23:29:20,810 INFO Request ID is eb5ec2b1-f233-4667-961f-a9e0b9b0046b
2026-08-03 23:29:21,006 INFO status has been updated to accepted
2026-08-03 23:29:42,965 INFO status has been updated to running
2026-08-03 23:35:42,590 INFO status has been updated to successful


...

Completed: era5_land_manitoba_2024_08_hourly.nc (93.13 MB)


2026-08-03 23:44:20,905 INFO Request ID is 8c2a77d6-fb22-4ce0-8f74-60c57d0ea4f9
2026-08-03 23:44:21,069 INFO status has been updated to accepted
2026-08-03 23:44:54,502 INFO status has been updated to running
2026-08-03 23:52:43,105 INFO status has been updated to successful


Completed: era5_land_manitoba_2024_09_hourly.nc (81.17 MB)


2026-08-03 23:52:50,261 INFO Request ID is bd34012a-fb7b-43e5-8af1-67984783ed65
2026-08-03 23:52:50,478 INFO status has been updated to accepted
2026-08-03 23:53:12,401 INFO status has been updated to running
2026-08-04 00:03:16,115 INFO status has been updated to successful
                                                                                         

Completed: era5_land_manitoba_2024_10_hourly.nc (88.85 MB)


## Download the November 1 time-zone buffer

October 31 in Manitoba requires several UTC hours from November 1.

In [6]:
november_buffer_path = (
    weather_folder
    / "era5_land_manitoba_2024_11_01_hourly.nc"
)

if (
    november_buffer_path.exists()
    and november_buffer_path.stat().st_size > 1_000_000
):
    print(
        "November buffer already available:",
        november_buffer_path.name,
    )

else:
    if november_buffer_path.exists():
        november_buffer_path.unlink()

    november_request = {
        "variable": VARIABLES,
        "year": "2024",
        "month": "11",
        "day": ["01"],
        "time": TIMES,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": MANITOBA_AREA,
    }

    print("Downloading November 1 buffer...")

    client.retrieve(
        DATASET_NAME,
        november_request,
        str(november_buffer_path),
    )

    print(
        "Downloaded:",
        november_buffer_path.name,
    )

print(
    "Buffer size:",
    round(
        november_buffer_path.stat().st_size
        / 1_000_000,
        2,
    ),
    "MB",
)

2026-08-04 00:05:30,688 INFO Request ID is b134121f-34e3-45b8-b985-a3077a26af29
2026-08-04 00:05:30,870 INFO status has been updated to accepted
2026-08-04 00:06:05,580 INFO status has been updated to running
2026-08-04 00:07:27,367 INFO status has been updated to successful
                                                                                        

Downloaded: era5_land_manitoba_2024_11_01_hourly.nc
Buffer size: 2.54 MB


## Verify all required files

In [7]:
required_files = [
    monthly_hourly_paths[month]
    for month in FIRE_SEASON_MONTHS
]

required_files.append(
    november_buffer_path
)

verification_rows = []

for path in required_files:
    exists = path.exists()

    size_mb = (
        path.stat().st_size / 1_000_000
        if exists
        else 0
    )

    verification_rows.append(
        {
            "FILE": path.name,
            "EXISTS": exists,
            "SIZE_MB": round(
                size_mb,
                2,
            ),
        }
    )

verification_table = pd.DataFrame(
    verification_rows
)

display(verification_table)

missing_files = verification_table.loc[
    ~verification_table["EXISTS"]
]

very_small_files = verification_table.loc[
    verification_table["SIZE_MB"] < 1
]

print(
    "\nRequired files:",
    len(verification_table),
)

print(
    "Missing files:",
    len(missing_files),
)

print(
    "Files smaller than 1 MB:",
    len(very_small_files),
)

,FILE,EXISTS,SIZE_MB
0,era5_land_manitoba_2024_04_hourly.nc,True,86.22
1,era5_land_manitoba_2024_05_hourly.nc,True,93.89
2,era5_land_manitoba_2024_06_hourly.nc,True,87.32
3,era5_land_manitoba_2024_07_hourly.nc,True,93.53
4,era5_land_manitoba_2024_08_hourly.nc,True,93.13
5,era5_land_manitoba_2024_09_hourly.nc,True,81.17
6,era5_land_manitoba_2024_10_hourly.nc,True,88.85
7,era5_land_manitoba_2024_11_01_hourly.nc,True,2.54



Required files: 8
Missing files: 0
Files smaller than 1 MB: 0


## Open all hourly files

In [8]:
hourly_weather_raw = xr.open_mfdataset(
    required_files,
    combine="by_coords",
    chunks={"valid_time": 168},
)

hourly_weather_raw = hourly_weather_raw.sortby(
    "valid_time"
)

time_index = pd.DatetimeIndex(
    hourly_weather_raw["valid_time"].values
)

print("Hourly dimensions:")
print(dict(hourly_weather_raw.sizes))

print("\nFirst timestamp:", time_index.min())
print("Last timestamp:", time_index.max())
print("Hourly timestamps:", len(time_index))
print("Duplicate timestamps:", time_index.duplicated().sum())

print("\nVariables:")
print(list(hourly_weather_raw.data_vars))

Hourly dimensions:
{'valid_time': 5160, 'latitude': 115, 'longitude': 129}

First timestamp: 2024-04-01 00:00:00
Last timestamp: 2024-11-01 23:00:00
Hourly timestamps: 5160
Duplicate timestamps: 0

Variables:
['t2m', 'd2m', 'u10', 'v10', 'tp']


/var/folders/dk/g6kdw5nj5hb8tlfg5r6rfn300000gn/T/ipykernel_87311/3891183770.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 168. This could degrade performance. Instead, consider rechunking after loading.
  hourly_weather_raw = xr.open_mfdataset(
/var/folders/dk/g6kdw5nj5hb8tlfg5r6rfn300000gn/T/ipykernel_87311/3891183770.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 168. This could degrade performance. Instead, consider rechunking after loading.
  hourly_weather_raw = xr.open_mfdataset(
/var/folders/dk/g6kdw5nj5hb8tlfg5r6rfn300000gn/T/ipykernel_87311/3891183770.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 168. This could degrade performance. Instead, consider rechunking after loading.
  hourly_weather_raw = xr.open_mfdataset(
/var/folders/dk/g6kdw5nj5hb8tlfg5r6rfn300000gn/T/ipykernel_87

## Create hourly weather variables

In [9]:
hourly_weather = xr.Dataset(
    {
        "TEMPERATURE_C": (
            hourly_weather_raw["t2m"] - 273.15
        ),
        "DEWPOINT_C": (
            hourly_weather_raw["d2m"] - 273.15
        ),
        "WIND_SPEED_MS": np.sqrt(
            hourly_weather_raw["u10"] ** 2
            + hourly_weather_raw["v10"] ** 2
        ),
        "PRECIPITATION_MM": (
            hourly_weather_raw["tp"] * 1000
        ),
    }
)

hourly_weather["RELATIVE_HUMIDITY_PCT"] = (
    100
    * np.exp(
        (
            17.625
            * hourly_weather["DEWPOINT_C"]
            / (
                243.04
                + hourly_weather["DEWPOINT_C"]
            )
        )
        -
        (
            17.625
            * hourly_weather["TEMPERATURE_C"]
            / (
                243.04
                + hourly_weather["TEMPERATURE_C"]
            )
        )
    )
).clip(min=0, max=100)

print("Created variables:")
print(list(hourly_weather.data_vars))

Created variables:
['TEMPERATURE_C', 'DEWPOINT_C', 'WIND_SPEED_MS', 'PRECIPITATION_MM', 'RELATIVE_HUMIDITY_PCT']


## Convert UTC timestamps to Manitoba local dates

In [10]:
utc_times = pd.DatetimeIndex(
    hourly_weather["valid_time"].values
).tz_localize("UTC")

local_times = utc_times.tz_convert(
    "America/Winnipeg"
)

instantaneous_dates = (
    local_times
    .tz_localize(None)
    .normalize()
    .to_numpy(dtype="datetime64[ns]")
)

# Precipitation represents the interval ending at its timestamp.
precipitation_dates = (
    (
        local_times
        - pd.Timedelta(nanoseconds=1)
    )
    .tz_localize(None)
    .normalize()
    .to_numpy(dtype="datetime64[ns]")
)

print("First UTC timestamp:", utc_times[0])
print("First Manitoba timestamp:", local_times[0])

print("\nLast UTC timestamp:", utc_times[-1])
print("Last Manitoba timestamp:", local_times[-1])

First UTC timestamp: 2024-04-01 00:00:00+00:00
First Manitoba timestamp: 2024-03-31 19:00:00-05:00

Last UTC timestamp: 2024-11-01 23:00:00+00:00
Last Manitoba timestamp: 2024-11-01 18:00:00-05:00


## Aggregate hourly values into daily weather

In [11]:
hourly_weather = hourly_weather.assign_coords(
    LOCAL_DATE=(
        "valid_time",
        instantaneous_dates,
    )
)

precipitation_hourly = (
    hourly_weather["PRECIPITATION_MM"]
    .assign_coords(
        PRECIPITATION_DATE=(
            "valid_time",
            precipitation_dates,
        )
    )
)

daily_precipitation = (
    precipitation_hourly
    .groupby("PRECIPITATION_DATE")
    .sum("valid_time")
    .rename(
        {
            "PRECIPITATION_DATE":
            "LOCAL_DATE"
        }
    )
)

daily_weather = xr.Dataset(
    {
        "TEMP_MAX_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .max("valid_time")
        ),
        "TEMP_MIN_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .min("valid_time")
        ),
        "TEMP_MEAN_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "RH_MIN_PCT": (
            hourly_weather[
                "RELATIVE_HUMIDITY_PCT"
            ]
            .groupby("LOCAL_DATE")
            .min("valid_time")
        ),
        "RH_MEAN_PCT": (
            hourly_weather[
                "RELATIVE_HUMIDITY_PCT"
            ]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "WIND_MAX_MS": (
            hourly_weather["WIND_SPEED_MS"]
            .groupby("LOCAL_DATE")
            .max("valid_time")
        ),
        "WIND_MEAN_MS": (
            hourly_weather["WIND_SPEED_MS"]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "PRECIPITATION_MM": (
            daily_precipitation
        ),
    }
)

print("Daily dimensions before filtering:")
print(dict(daily_weather.sizes))

Daily dimensions before filtering:
{'latitude': 115, 'longitude': 129, 'LOCAL_DATE': 216}


## Verify every fire-season date has 24 hours

In [12]:
season_dates = pd.date_range(
    start="2024-04-01",
    end="2024-10-31",
    freq="D",
)

instantaneous_counts = pd.Series(
    instantaneous_dates
).value_counts()

precipitation_counts = pd.Series(
    precipitation_dates
).value_counts()

incomplete_dates = []

for date in season_dates:
    instantaneous_hours = (
        instantaneous_counts.get(
            date,
            0,
        )
    )

    precipitation_hours = (
        precipitation_counts.get(
            date,
            0,
        )
    )

    if (
        instantaneous_hours != 24
        or precipitation_hours != 24
    ):
        incomplete_dates.append(
            {
                "LOCAL_DATE": date,
                "INSTANTANEOUS_HOURS":
                    instantaneous_hours,
                "PRECIPITATION_HOURS":
                    precipitation_hours,
            }
        )

print("Expected fire-season dates:", len(season_dates))
print("Incomplete dates:", len(incomplete_dates))

if incomplete_dates:
    display(
        pd.DataFrame(incomplete_dates)
    )

Expected fire-season dates: 214
Incomplete dates: 0


## Keep April through October only

In [13]:
season_date_values = season_dates.to_numpy(
    dtype="datetime64[ns]"
)

daily_weather_2024 = daily_weather.sel(
    LOCAL_DATE=season_date_values
).load()

hourly_weather_raw.close()

print("Fire-season daily dimensions:")
print(dict(daily_weather_2024.sizes))

print("\nFirst date:")
print(
    daily_weather_2024[
        "LOCAL_DATE"
    ].values[0]
)

print("\nLast date:")
print(
    daily_weather_2024[
        "LOCAL_DATE"
    ].values[-1]
)

Fire-season daily dimensions:
{'latitude': 115, 'longitude': 129, 'LOCAL_DATE': 214}

First date:
2024-04-01T00:00:00.000000000

Last date:
2024-10-31T00:00:00.000000000


## Save the daily gridded weather

In [14]:
daily_weather_path = (
    weather_folder
    / (
        "era5_land_manitoba_"
        "2024_fire_season_daily.nc"
    )
)

daily_weather_2024.to_netcdf(
    daily_weather_path
)

print("Saved:", daily_weather_path)
print("Exists:", daily_weather_path.exists())

print(
    "Size:",
    round(
        daily_weather_path.stat().st_size
        / 1_000_000,
        2,
    ),
    "MB",
)

Saved: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/2024/era5_land_manitoba_2024_fire_season_daily.nc
Exists: True
Size: 101.66 MB


## Load the validated ERA5 matching table

In [15]:
era5_match_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_grid_to_era5_land_matches.parquet"
)

era5_matches = pd.read_parquet(
    era5_match_path
)

print("ERA5 match rows:", len(era5_matches))

print(
    "Duplicate GRID_ID values:",
    era5_matches[
        "GRID_ID"
    ].duplicated().sum(),
)

display(
    era5_matches.head()
)

ERA5 match rows: 6501
Duplicate GRID_ID values: 0


,GRID_ID,CENTER_LATITUDE,CENTER_LONGITUDE,MB_AREA_KM2,MB_COVERAGE_PCT,ERA5_LATITUDE,ERA5_LONGITUDE,ERA5_MATCH_DISTANCE_KM,ERA5_MATCH_QUALITY
0,MB_005_000,49.034036,-101.408339,14.489028,14.489028,49.0,-101.4,3.833177,Near: 0–10 km
1,MB_006_000,49.122970,-101.428824,2.229798,2.229798,49.1,-101.4,3.305338,Near: 0–10 km
2,MB_007_000,49.211928,-101.449396,7.044500,7.044500,49.2,-101.4,3.825834,Near: 0–10 km
3,MB_008_000,49.300912,-101.470056,0.002842,0.002842,49.3,-101.5,2.173562,Near: 0–10 km
4,MB_004_001,48.958440,-101.252939,4.265625,4.265625,49.0,-101.3,5.757781,Near: 0–10 km


## Assign daily weather to every Manitoba grid cell

In [16]:
grid_ids = era5_matches[
    "GRID_ID"
].to_numpy()

latitude_indexer = xr.DataArray(
    era5_matches[
        "ERA5_LATITUDE"
    ].to_numpy(),
    dims="GRID_ID",
    coords={
        "GRID_ID": grid_ids
    },
)

longitude_indexer = xr.DataArray(
    era5_matches[
        "ERA5_LONGITUDE"
    ].to_numpy(),
    dims="GRID_ID",
    coords={
        "GRID_ID": grid_ids
    },
)

weather_at_grid = daily_weather_2024.sel(
    latitude=latitude_indexer,
    longitude=longitude_indexer,
    method="nearest",
)

print("Weather-at-grid dimensions:")
print(dict(weather_at_grid.sizes))

Weather-at-grid dimensions:
{'GRID_ID': 6501, 'LOCAL_DATE': 214}


## Convert the weather data into a table

In [17]:
weather_grid_df = (
    weather_at_grid
    .to_dataframe()
    .reset_index()
    .rename(
        columns={
            "latitude":
                "ERA5_LATITUDE",
            "longitude":
                "ERA5_LONGITUDE",
        }
    )
)

weather_grid_df = weather_grid_df.drop(
    columns=[
        "number",
        "expver",
    ],
    errors="ignore",
)

weather_grid_df["LOCAL_DATE"] = (
    pd.to_datetime(
        weather_grid_df["LOCAL_DATE"]
    )
    .dt.normalize()
)

match_metadata = [
    "GRID_ID",
    "CENTER_LATITUDE",
    "CENTER_LONGITUDE",
    "MB_AREA_KM2",
    "MB_COVERAGE_PCT",
    "ERA5_MATCH_DISTANCE_KM",
    "ERA5_MATCH_QUALITY",
]

weather_grid_df = weather_grid_df.merge(
    era5_matches[match_metadata],
    on="GRID_ID",
    how="left",
    validate="many_to_one",
)

expected_rows = (
    len(season_dates)
    * len(era5_matches)
)

print("Expected rows:", expected_rows)
print("Actual rows:", len(weather_grid_df))

print(
    "Duplicate date-grid rows:",
    weather_grid_df.duplicated(
        subset=[
            "LOCAL_DATE",
            "GRID_ID",
        ]
    ).sum(),
)

Expected rows: 1391214
Actual rows: 1391214
Duplicate date-grid rows: 0


## Validate weather completeness

In [18]:
weather_variables = [
    "TEMP_MAX_C",
    "TEMP_MIN_C",
    "TEMP_MEAN_C",
    "RH_MIN_PCT",
    "RH_MEAN_PCT",
    "WIND_MAX_MS",
    "WIND_MEAN_MS",
    "PRECIPITATION_MM",
]

missing_weather = (
    weather_grid_df[
        weather_variables
    ]
    .isna()
    .sum()
    .to_frame("MISSING_COUNT")
)

display(missing_weather)

print(
    "Total missing weather values:",
    missing_weather[
        "MISSING_COUNT"
    ].sum(),
)

,MISSING_COUNT
TEMP_MAX_C,0
TEMP_MIN_C,0
TEMP_MEAN_C,0
RH_MIN_PCT,0
RH_MEAN_PCT,0
WIND_MAX_MS,0
WIND_MEAN_MS,0
PRECIPITATION_MM,0


Total missing weather values: 0


## Load and prepare wildfire targets

In [19]:
targets_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_daily_fire_targets_2005_2025.parquet"
)

fire_targets = pd.read_parquet(
    targets_path
)

fire_targets["FIRE_DATE"] = (
    pd.to_datetime(
        fire_targets["FIRE_DATE"]
    )
    .dt.normalize()
)

target_fields = [
    "FIRE_DATE",
    "GRID_ID",
    "FIRE_OCCURRED",
    "FIRE_COUNT",
    "NATURAL_FIRE_COUNT",
    "HUMAN_FIRE_COUNT",
    "UNKNOWN_FIRE_COUNT",
    "TOTAL_BURNED_HA",
    "MAX_FIRE_SIZE_HA",
]

fire_season_targets = fire_targets.loc[
    fire_targets["FIRE_DATE"].between(
        "2024-04-01",
        "2024-10-31",
    ),
    target_fields,
].copy()

print(
    "Positive target rows:",
    len(fire_season_targets),
)

print(
    "Individual fires:",
    fire_season_targets[
        "FIRE_COUNT"
    ].sum(),
)

Positive target rows: 298
Individual fires: 312


## Combine weather and wildfire targets

In [20]:
model_table_2024 = weather_grid_df.merge(
    fire_season_targets,
    left_on=[
        "LOCAL_DATE",
        "GRID_ID",
    ],
    right_on=[
        "FIRE_DATE",
        "GRID_ID",
    ],
    how="left",
    validate="one_to_one",
)

count_columns = [
    "FIRE_OCCURRED",
    "FIRE_COUNT",
    "NATURAL_FIRE_COUNT",
    "HUMAN_FIRE_COUNT",
    "UNKNOWN_FIRE_COUNT",
]

model_table_2024[count_columns] = (
    model_table_2024[count_columns]
    .fillna(0)
    .astype("int16")
)

size_columns = [
    "TOTAL_BURNED_HA",
    "MAX_FIRE_SIZE_HA",
]

model_table_2024[size_columns] = (
    model_table_2024[size_columns]
    .fillna(0.0)
)

model_table_2024 = (
    model_table_2024
    .drop(
        columns="FIRE_DATE"
    )
    .sort_values(
        [
            "LOCAL_DATE",
            "GRID_ID",
        ]
    )
    .reset_index(drop=True)
)

print("Model-table rows:", len(model_table_2024))

print(
    "Positive grid-cell days:",
    model_table_2024[
        "FIRE_OCCURRED"
    ].sum(),
)

print(
    "Individual fires represented:",
    model_table_2024[
        "FIRE_COUNT"
    ].sum(),
)

print(
    "Duplicate date-grid rows:",
    model_table_2024.duplicated(
        subset=[
            "LOCAL_DATE",
            "GRID_ID",
        ]
    ).sum(),
)

Model-table rows: 1391214
Positive grid-cell days: 298
Individual fires represented: 312
Duplicate date-grid rows: 0


## Save the complete 2024 tables

In [21]:
weather_output_path = (
    project_root
    / "data"
    / "interim"
    / "era5_grid_weather_2024_fire_season.parquet"
)

model_output_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_model_table_2024_fire_season.parquet"
)

weather_grid_df.to_parquet(
    weather_output_path,
    index=False,
)

model_table_2024.to_parquet(
    model_output_path,
    index=False,
)

for path in [
    weather_output_path,
    model_output_path,
]:
    print("\nSaved:", path.name)
    print("Exists:", path.exists())

    print(
        "Size:",
        round(
            path.stat().st_size
            / 1_000_000,
            2,
        ),
        "MB",
    )


Saved: era5_grid_weather_2024_fire_season.parquet
Exists: True
Size: 47.43 MB

Saved: manitoba_model_table_2024_fire_season.parquet
Exists: True
Size: 57.37 MB
